# AAP 2.7 y Vault en VMs RHEL 9: AppRole y OIDC

Este notebook configura y evalúa **credenciales externas nativas de AAP** contra Vault. El playbook no autentica por su cuenta: AAP obtiene el secreto e inyecta `AAP_VAULT_DEMO_SECRET` en el job. OIDC utiliza la identidad del workload emitida por AAP; no configura SSO de usuarios.

Todas las operaciones se muestran en celdas `%%bash`, con `vault`, `curl`, `jq` y `ssh`. Para copiar una celda a una terminal Bash, situarse en `vm-rhel9/aap` y quitar `%%bash`. `aap-env.sh` solo define variables y argumentos de conexión. Los secretos se leen de `../.state/aap/`, excluido de Git.

El notebook se conserva aquí con sus resultados de ejecución. `evaluation.json` es el resumen de pruebas, no un segundo conjunto de notebooks.

## Instalación y prerrequisitos

La VM dedicada se instala con el [módulo Terraform](../terraform/aap/) y el [instalador Bash completo](install.sh). Desde `vm-rhel9`:

```bash
bash aap/deploy.sh
bash aap/install.sh "$HOME/Downloads/ansible-automation-platform-containerized-setup-bundle-2.7-1.1-x86_64.tar.gz"
```

Topología Growth: RHEL 9, 8 vCPU, 32 GiB, 150 GiB, Podman rootless y SELinux Enforcing. El inventario contiene `automationgateway`, `automationcontroller`, `automationhub`, `automationeda`, `automationmetrics` y `database` en el mismo host. La configuración del instalador incluye:

```yaml
bundle_install: true
bundle_dir: /home/ec2-user/aap/installer/bundle
redis_mode: standalone
hub_seed_collections: false
controller_percent_memory_capacity: 0.5
feature_flags:
  FEATURE_OIDC_WORKLOAD_IDENTITY_ENABLED: true
```

Se necesitan el bundle oficial de Red Hat y el manifiesto en `../.state/aap/manifest.zip`. Las contraseñas PostgreSQL y admin se generan y guardan en esa carpeta privada. El servicio de métricas tiene credenciales distintas para su base de datos y para leer la base de datos del controller.

La siguiente celda comprueba la instalación terminada antes de configurar la integración. El acceso de administración está descrito en [README.md](README.md). OIDC para Vault es Technology Preview en AAP 2.7.


In [1]:
%%bash
set -euo pipefail
source ./aap-env.sh
curl -fsS "$AAP_API/ping/" | jq '{version,active_node}'
ssh -n "${SSH_ARGS[@]}" "ec2-user@$AAP_IP" 'set -e; test "$(cat ~/aap/install.exit)" = 0; test "$(getenforce)" = Enforcing; cat /etc/redhat-release; nproc; podman ps --format "{{.Names}} {{.Status}}"'


{


  "version": "4.8.0",


  "active_node": "aap-vm.jose-merchan.sbx.hashidemos.io"


}


Red Hat Enterprise Linux release 9.8 (Plow)


...

automation-hub-worker-1 Up 19 hours


automation-hub-worker-2 Up 19 hours


automation-metrics-web Up 19 hours


automation-metrics-tasks Up 19 hours


automation-metrics-scheduler Up 19 hours


## 1. Autenticación de administración de la API

Se crea un token mediante la CLI oficial en la VM, o se reutiliza la sesión privada válida. El token administrativo solo configura AAP; no se utiliza para autenticar los jobs con Vault.

In [2]:
%%bash
set -euo pipefail
source ./aap-env.sh
if [[ -s "$STATE/aap/api.curl" ]] && "${AAP_CURL[@]}" "$AAP_GATEWAY_API/me/" > "$STATE/aap/me.json" 2>/dev/null; then
  echo 'AAP API session already valid'
  exit 0
fi
ssh -n "${SSH_ARGS[@]}" "ec2-user@$AAP_IP" \
  'podman exec automation-gateway aap-gateway-manage create_oauth2_token --user admin --no-color' > "$STATE/aap/token-command.txt"
awk 'NF == 1 && /^[[:alnum:]_-]+$/ {print; exit} /New OAuth2 token for admin:/ {print $NF; exit}' "$STATE/aap/token-command.txt" > "$STATE/aap/api-token"
TOKEN=$(cat "$STATE/aap/api-token")
if [[ ! "$TOKEN" =~ ^[A-Za-z0-9_-]{20,}$ ]]; then
  echo 'The management command did not return a valid token' >&2
  exit 1
fi
printf 'header = "Authorization: Bearer %s"\n' "$TOKEN" > "$STATE/aap/api.curl"
unset TOKEN
"${AAP_CURL[@]}" "$AAP_GATEWAY_API/me/" > "$STATE/aap/me.json"
echo 'AAP API session created; token saved only in .state/aap/'


AAP API session already valid


## 2. Activar la suscripción

La API importa el manifiesto y comprueba que la licencia es válida. El ZIP y su contenido codificado nunca se imprimen.

In [3]:
%%bash
set -euo pipefail
source ./aap-env.sh
test -s "$STATE/aap/manifest.zip"
test -s "$STATE/aap/api.curl"
base64 < "$STATE/aap/manifest.zip" | tr -d '\n' > "$STATE/aap/manifest.base64"
jq -n --rawfile manifest "$STATE/aap/manifest.base64" '{manifest:$manifest}' > "$STATE/aap/activation-payload.json"
"${AAP_CURL[@]}" -X POST --data @"$STATE/aap/activation-payload.json" "$AAP_API/config/" > "$STATE/aap/activation.json"
"${AAP_CURL[@]}" "$AAP_API/config/" > "$STATE/aap/config.json"
jq -e '.license_info.valid_key == true' "$STATE/aap/config.json" >/dev/null
jq '.license_info | {license_type,instance_count,valid_key,trial,subscription_name,time_remaining}' "$STATE/aap/config.json"


{


  "license_type": "trial",


  "instance_count": 100,


  "valid_key": true,


  "trial": true,


  "subscription_name": "60 Day Product Trial of Red Hat Ansible Automation Platform, Self-Supported (100 Managed Nodes)",


  "time_remaining": 5070415


}


## 3. Vault KV, política y AppRole

Se crea un secreto de demo y una política limitada a una única ruta. El test verifica lectura, rechazo 403 fuera de esa ruta y revocación del token temporal. El SecretID dura 30 días; al caducar, volver a ejecutar esta celda y la de credenciales AAP.

In [4]:
%%bash
set -euo pipefail
source ../scripts/notebook-env.sh
mkdir -p "$STATE/aap"
vault secrets list -format=json | jq -e 'has("aap-demo/")' >/dev/null || vault secrets enable -path=aap-demo kv-v2
for attempt in $(seq 1 30); do vault read aap-demo/config >/dev/null 2>&1 && break; sleep 1; done
if ! vault kv get -format=json aap-demo/credentials/test > "$STATE/aap/demo-secret.json" 2>/dev/null; then
  openssl rand -hex 32 | tr -d '\n' > "$STATE/aap/demo-password"
  vault kv put aap-demo/credentials/test password=@"$STATE/aap/demo-password" purpose=aap-vm-demo >/dev/null
  vault kv get -format=json aap-demo/credentials/test > "$STATE/aap/demo-secret.json"
fi
jq -jr .data.data.password "$STATE/aap/demo-secret.json" | shasum -a 256 | awk '{print $1}' > "$STATE/aap/expected-digest"
vault policy write aap-demo-read - <<'HCL'
path "aap-demo/data/credentials/test" {
  capabilities = ["read"]
}
HCL
vault auth list -format=json | jq -e 'has("aap-approle/")' >/dev/null || vault auth enable -path=aap-approle approle
vault write auth/aap-approle/role/aap-demo \
  token_policies=aap-demo-read token_ttl=15m token_max_ttl=30m \
  secret_id_ttl=720h secret_id_num_uses=0 >/dev/null
vault read -format=json auth/aap-approle/role/aap-demo/role-id > "$STATE/aap/role-id.json"
if [[ ! -s "$STATE/aap/secret-id.json" ]] || ! vault write auth/aap-approle/role/aap-demo/secret-id-accessor/lookup \
    secret_id_accessor="$(jq -r .data.secret_id_accessor "$STATE/aap/secret-id.json")" >/dev/null 2>&1; then
  vault write -f -format=json auth/aap-approle/role/aap-demo/secret-id > "$STATE/aap/secret-id.json"
fi
jq -n --arg role "$(jq -r .data.role_id "$STATE/aap/role-id.json")" \
  --arg secret "$(jq -r .data.secret_id "$STATE/aap/secret-id.json")" \
  '{role_id:$role,secret_id:$secret}' > "$STATE/aap/approle-login-payload.json"
# Exercise the same application endpoint used by AAP external credentials.
export VAULT_ADDR="${VAULT_APPLICATION_ADDR:?Deploy the application FQDN first}"
curl -fsS -H 'Content-Type: application/json' --data @"$STATE/aap/approle-login-payload.json" \
  "$VAULT_ADDR/v1/auth/aap-approle/login" > "$STATE/aap/approle-login.json"
TOKEN=$(jq -er .auth.client_token "$STATE/aap/approle-login.json")
trap 'VAULT_TOKEN="$TOKEN" vault token revoke -self >/dev/null' EXIT
VAULT_TOKEN="$TOKEN" vault kv get -field=password aap-demo/credentials/test > "$STATE/aap/approle-read"
[[ "$(tr -d '\n' < "$STATE/aap/approle-read" | shasum -a 256 | awk '{print $1}')" == "$(cat "$STATE/aap/expected-digest")" ]]
[[ "$(VAULT_TOKEN="$TOKEN" vault token capabilities aap-demo/data/outside-demo)" == deny ]]
if VAULT_TOKEN="$TOKEN" vault kv get aap-demo/outside-demo > "$STATE/aap/denied.log" 2>&1; then
  echo 'Unexpected access outside the demo policy' >&2; exit 1
fi
grep -q '403' "$STATE/aap/denied.log"
echo 'AppRole: login, authorized read, denied read and token revocation verified'


Success! Uploaded policy: aap-demo-read


AppRole: login, authorized read, denied read and token revocation verified


## 4. Proyecto, credencial inyectada y plantillas AAP

El proyecto público contiene el playbook de comprobación. La credencial personalizada recibe el valor mediante un input source externo. El playbook compara su SHA-256 con `no_log: true`. Los objetos se reconcilian por nombre, sin crear duplicados.

In [5]:
%%bash
set -euo pipefail
source ./aap-env.sh

# Reconcile only the dedicated demo objects. API responses stay outside Git.
upsert() {
  local resource=$1 name=$2 payload=$3 id
  "${AAP_CURL[@]}" --get --data-urlencode "name=$name" "$AAP_API/$resource/" > "$STATE/aap/search.json"
  jq -e '.count <= 1' "$STATE/aap/search.json" >/dev/null
  id=$(jq -r '.results[0].id // empty' "$STATE/aap/search.json")
  if [[ -n "$id" ]]; then
    "${AAP_CURL[@]}" -X PATCH --data @"$payload" "$AAP_API/$resource/$id/" > "$STATE/aap/object.json"
  else
    "${AAP_CURL[@]}" -X POST --data @"$payload" "$AAP_API/$resource/" > "$STATE/aap/object.json"
  fi
  jq -er .id "$STATE/aap/object.json"
}

"${AAP_CURL[@]}" --get --data-urlencode name=Default "$AAP_API/organizations/" > "$STATE/aap/organizations.json"
ORG=$(jq -er '.results[0].id' "$STATE/aap/organizations.json")
"${AAP_CURL[@]}" "$AAP_API/execution_environments/?page_size=200" > "$STATE/aap/execution-environments.json"
EE=$(jq -er '[.results[] | select(.name | test("Default execution environment|supported";"i"))][0].id' "$STATE/aap/execution-environments.json")

jq -n --argjson org "$ORG" '{name:"Vault VM demo inventory",organization:$org,variables:"ansible_connection: local"}' > "$STATE/aap/inventory-payload.json"
INVENTORY=$(upsert inventories 'Vault VM demo inventory' "$STATE/aap/inventory-payload.json")
"${AAP_CURL[@]}" "$AAP_API/inventories/$INVENTORY/hosts/?name=localhost" > "$STATE/aap/hosts.json"
if [[ $(jq -r .count "$STATE/aap/hosts.json") == 0 ]]; then
  "${AAP_CURL[@]}" -X POST -d '{"name":"localhost","variables":"ansible_connection: local"}' "$AAP_API/inventories/$INVENTORY/hosts/" > "$STATE/aap/host.json"
fi

jq -n --argjson org "$ORG" '{name:"Vault VM demo project",organization:$org,scm_type:"git",scm_url:"https://github.com/jm-merchan/Vault_Use_Cases_Example_202607.git",scm_branch:"codex/aap-vm-vault",scm_update_on_launch:false}' > "$STATE/aap/project-payload.json"
PROJECT=$(upsert projects 'Vault VM demo project' "$STATE/aap/project-payload.json")
for attempt in $(seq 1 120); do
  "${AAP_CURL[@]}" "$AAP_API/projects/$PROJECT/" > "$STATE/aap/project.json"
  status=$(jq -r .status "$STATE/aap/project.json")
  [[ "$status" != successful ]] || break
  [[ "$status" != failed && "$status" != error ]] || { echo "Project sync: $status" >&2; exit 1; }
  sleep 3
done
jq -e '.status == "successful"' "$STATE/aap/project.json" >/dev/null

cat > "$STATE/aap/credential-type-payload.json" <<'JSON'
{"name":"Vault VM demo secret","kind":"cloud","inputs":{"fields":[{"id":"vault_password","label":"Demo password from Vault","type":"string","secret":true}],"required":["vault_password"]},"injectors":{"env":{"AAP_VAULT_DEMO_SECRET":"{{ vault_password }}"}}}
JSON
TYPE=$(upsert credential_types 'Vault VM demo secret' "$STATE/aap/credential-type-payload.json")

jq -n --argjson organization "$ORG" --argjson inventory "$INVENTORY" --argjson project "$PROJECT" --argjson ee "$EE" --argjson type "$TYPE" \
  '{organization:$organization,inventory:$inventory,project:$project,execution_environment:$ee,credential_type:$type}' > "$STATE/aap/controller-resources.json"

for method in AppRole OIDC; do
  jq -n --arg name "Vault VM demo $method secret" --argjson org "$ORG" --argjson type "$TYPE" \
    '{name:$name,organization:$org,credential_type:$type,inputs:{}}' > "$STATE/aap/target-credential-payload.json"
  CREDENTIAL=$(upsert credentials "Vault VM demo $method secret" "$STATE/aap/target-credential-payload.json")
  jq -n --arg name "Vault VM - $method" --argjson inventory "$INVENTORY" --argjson project "$PROJECT" --argjson ee "$EE" \
    --arg method "$method" --rawfile digest "$STATE/aap/expected-digest" \
    '{name:$name,job_type:"run",inventory:$inventory,project:$project,execution_environment:$ee,
      playbook:"vm-rhel9/aap/playbooks/verify-secret.yml",timeout:600,
      extra_vars:({aap_auth_method:$method,aap_expected_digest:($digest|rtrimstr("\n"))}|tojson)}' > "$STATE/aap/job-template-payload.json"
  TEMPLATE=$(upsert job_templates "Vault VM - $method" "$STATE/aap/job-template-payload.json")
  "${AAP_CURL[@]}" "$AAP_API/job_templates/$TEMPLATE/credentials/" > "$STATE/aap/template-credentials.json"
  if ! jq -e --argjson id "$CREDENTIAL" 'any(.results[]; .id == $id)' "$STATE/aap/template-credentials.json" >/dev/null; then
    jq -n --argjson id "$CREDENTIAL" '{id:$id}' > "$STATE/aap/associate-payload.json"
    "${AAP_CURL[@]}" -X POST --data @"$STATE/aap/associate-payload.json" "$AAP_API/job_templates/$TEMPLATE/credentials/" > "$STATE/aap/associate.json"
  fi
  jq --arg method "$method" --argjson credential "$CREDENTIAL" --argjson template "$TEMPLATE" \
    '.[$method]={credential:$credential,job_template:$template}' "$STATE/aap/controller-resources.json" > "$STATE/aap/controller-resources.next.json"
  mv "$STATE/aap/controller-resources.next.json" "$STATE/aap/controller-resources.json"
done
jq . "$STATE/aap/controller-resources.json"


{


  "organization": 1,


  "inventory": 2,


  "project": 7,


  "execution_environment": 4,


...

  "OIDC": {


    "credential": 4,


    "job_template": 9


  }


}


## 5. Confianza OIDC de Vault en AAP

El gateway del bundle 2.7-1.1 emite `iss` con `/o/`, mientras discovery publica `/o`. Se fija el issuer exacto emitido. Los IDs de las claims son números JSON: `--argjson` conserva su tipo. La audiencia coincide con la URL de Vault y se restringen organización, proyecto y plantilla.

In [6]:
%%bash
set -euo pipefail
source ./aap-env.sh
curl -fsS "$AAP_ADDR/o/.well-known/openid-configuration/" > "$STATE/aap/oidc-discovery.json"
ISSUER=$(jq -er .issuer "$STATE/aap/oidc-discovery.json")
[[ "$ISSUER" == "$AAP_ADDR/o" ]]
curl -fsS "$(jq -er .jwks_uri "$STATE/aap/oidc-discovery.json")" > "$STATE/aap/jwks.json"
jq -e '.keys | length > 0' "$STATE/aap/jwks.json" >/dev/null

vault auth list -format=json | jq -e 'has("aap-jwt/")' >/dev/null || vault auth enable -path=aap-jwt jwt
# The bundled gateway emits workload JWTs with /o/ while discovery advertises /o.
# Match the actual signed iss claim exactly; the discovery URL stays unchanged.
vault write auth/aap-jwt/config oidc_discovery_url="$ISSUER" bound_issuer="${ISSUER%/}/" >/dev/null
jq -n --arg audience "$VAULT_APPLICATION_ADDR" \
  --argjson template "$(jq -er .OIDC.job_template "$STATE/aap/controller-resources.json")" \
  --argjson organization "$(jq -er .organization "$STATE/aap/controller-resources.json")" \
  --argjson project "$(jq -er .project "$STATE/aap/controller-resources.json")" \
  '{role_type:"jwt",user_claim:"sub",bound_audiences:[$audience],bound_claims_type:"string",
    bound_claims:{aap_controller_job_template_id:$template,aap_controller_organization_id:$organization,aap_controller_project_id:$project},
    token_policies:["aap-demo-read"],token_ttl:300,token_max_ttl:600}' > "$STATE/aap/jwt-role-payload.json"
vault write auth/aap-jwt/role/aap-demo @"$STATE/aap/jwt-role-payload.json" >/dev/null
vault read -format=json auth/aap-jwt/role/aap-demo | jq '.data | {role_type,bound_audiences,bound_claims,token_policies,token_ttl}'


{


  "role_type": "jwt",


  "bound_audiences": [


    "https://vault-vm-apps.jose-merchan.sbx.hashidemos.io"


  ],


...

  "token_policies": [


    "aap-demo-read"


  ],


  "token_ttl": 300


}


## 6. Credenciales externas de AAP

AppRole usa RoleID y SecretID. OIDC usa JWT Role y la identidad del job, sin SecretID ni token Vault estático. Para KV v2, `secret_backend=aap-demo` y `secret_path=credentials/test`: no se añade `/data/` manualmente. El tipo OIDC requiere `default_auth_path` también en los metadatos del input source.

In [7]:
%%bash
set -euo pipefail
source ./aap-env.sh

"${AAP_CURL[@]}" "$AAP_API/credential_types/?page_size=200" > "$STATE/aap/credential-types.json"
APPROLE_TYPE=$(jq -er '.results[] | select(.namespace == "hashivault_kv") | .id' "$STATE/aap/credential-types.json")
OIDC_TYPE=$(jq -er '.results[] | select(.namespace == "hashivault-kv-oidc") | .id' "$STATE/aap/credential-types.json")
ORG=$(jq -er .organization "$STATE/aap/controller-resources.json")

for method in AppRole OIDC; do
  if [[ "$method" == AppRole ]]; then
    jq -n --arg url "$VAULT_APPLICATION_ADDR" --argjson org "$ORG" --argjson type "$APPROLE_TYPE" \
      --slurpfile role "$STATE/aap/role-id.json" --slurpfile secret "$STATE/aap/secret-id.json" \
      '{name:"Vault VM - AppRole backend",organization:$org,credential_type:$type,
        inputs:{url:$url,api_version:"v2",default_auth_path:"aap-approle",role_id:$role[0].data.role_id,secret_id:$secret[0].data.secret_id}}' > "$STATE/aap/backend-payload.json"
  else
    jq -n --arg url "$VAULT_APPLICATION_ADDR" --argjson org "$ORG" --argjson type "$OIDC_TYPE" \
      '{name:"Vault VM - OIDC backend",organization:$org,credential_type:$type,
        inputs:{url:$url,api_version:"v2",default_auth_path:"aap-jwt",jwt_role:"aap-demo"}}' > "$STATE/aap/backend-payload.json"
  fi
  "${AAP_CURL[@]}" --get --data-urlencode "name=Vault VM - $method backend" "$AAP_API/credentials/" > "$STATE/aap/backend-search.json"
  BACKEND=$(jq -r '.results[0].id // empty' "$STATE/aap/backend-search.json")
  if [[ -n "$BACKEND" ]]; then
    "${AAP_CURL[@]}" -X PATCH --data @"$STATE/aap/backend-payload.json" "$AAP_API/credentials/$BACKEND/" > "$STATE/aap/backend.json"
  else
    "${AAP_CURL[@]}" -X POST --data @"$STATE/aap/backend-payload.json" "$AAP_API/credentials/" > "$STATE/aap/backend.json"
    BACKEND=$(jq -er .id "$STATE/aap/backend.json")
  fi
  TARGET=$(jq -er --arg method "$method" '.[$method].credential' "$STATE/aap/controller-resources.json")
  jq -n --argjson source "$BACKEND" --arg method "$method" \
    '{source_credential:$source,input_field_name:"vault_password",metadata:({secret_backend:"aap-demo",secret_path:"credentials/test",secret_key:"password"} +
      (if $method == "OIDC" then {default_auth_path:"aap-jwt"} else {auth_path:"aap-approle"} end))}' > "$STATE/aap/input-source-payload.json"
  "${AAP_CURL[@]}" "$AAP_API/credentials/$TARGET/input_sources/" > "$STATE/aap/input-sources.json"
  INPUT=$(jq -r '.results[] | select(.input_field_name == "vault_password") | .id' "$STATE/aap/input-sources.json")
  if [[ -n "$INPUT" ]]; then
    "${AAP_CURL[@]}" -X PATCH --data @"$STATE/aap/input-source-payload.json" "$AAP_API/credential_input_sources/$INPUT/" > "$STATE/aap/input-source.json"
  else
    "${AAP_CURL[@]}" -X POST --data @"$STATE/aap/input-source-payload.json" "$AAP_API/credentials/$TARGET/input_sources/" > "$STATE/aap/input-source.json"
  fi
  jq --arg method "$method" --argjson backend "$BACKEND" '.[$method].backend=$backend' "$STATE/aap/controller-resources.json" > "$STATE/aap/controller-resources.next.json"
  mv "$STATE/aap/controller-resources.next.json" "$STATE/aap/controller-resources.json"
  printf '%s external lookup configured: backend %s -> target credential %s\n' "$method" "$BACKEND" "$TARGET"
done


AppRole external lookup configured: backend 5 -> target credential 3


OIDC external lookup configured: backend 6 -> target credential 4


## 7. Evaluación real: lectura, rotación y rechazo

Se lanzan cuatro jobs positivos (dos métodos antes y después de la rotación) y una copia no autorizada de la plantilla OIDC. El quinto job debe fallar; se comprueba además el error preciso de la claim para evitar aceptar un fallo genérico como prueba válida. La rotación afecta solo al secreto dedicado de esta demo.

In [8]:
%%bash
set -euo pipefail
source ./aap-env.sh
printf '[]\n' > "$STATE/aap/test-results.json"

run_job() {
  local template=$1 method=$2 phase=$3 expected=$4 job status
  "${AAP_CURL[@]}" -X POST -d '{}' "$AAP_API/job_templates/$template/launch/" > "$STATE/aap/launch.json"
  job=$(jq -er .job "$STATE/aap/launch.json")
  for attempt in $(seq 1 180); do
    "${AAP_CURL[@]}" "$AAP_API/jobs/$job/" > "$STATE/aap/job-$job.json"
    status=$(jq -r .status "$STATE/aap/job-$job.json")
    case "$status" in
      successful) jq -e '.event_processing_finished' "$STATE/aap/job-$job.json" >/dev/null && break;;
      failed|error|canceled) break;;
    esac
    sleep 3
  done
  "${AAP_CURL[@]}" "$AAP_API/jobs/$job/stdout/?format=txt" > "$STATE/aap/job-$job.txt"
  if [[ "$expected" == successful ]]; then
    [[ "$status" == successful ]]
    grep -q 'Vault VM secret retrieved and verified' "$STATE/aap/job-$job.txt"
  else
    [[ "$status" == failed || "$status" == error ]]
    # A generic job failure is not proof that the JWT restriction worked.
    jq -n --arg template "$template" \
      '{metadata:{secret_backend:"aap-demo",secret_path:"credentials/test",secret_key:"password",default_auth_path:"aap-jwt",job_template_id:$template}}' > "$STATE/aap/negative-test-payload.json"
    backend=$(jq -er .OIDC.backend "$STATE/aap/controller-resources.json")
    http_code=$(curl -sS --config "$STATE/aap/api.curl" -H 'Content-Type: application/json' -X POST \
      --data @"$STATE/aap/negative-test-payload.json" --output "$STATE/aap/negative-test.json" --write-out '%{http_code}' \
      "$AAP_API/credentials/$backend/test/")
    [[ "$http_code" == 400 ]]
    jq -r '.details.error_message' "$STATE/aap/negative-test.json" > "$STATE/aap/negative-explanation.txt"
    grep -q 'aap_controller_job_template_id' "$STATE/aap/negative-explanation.txt"
    grep -Eiq 'bound claim|bound_claim|claim.*match|claim.*valid' "$STATE/aap/negative-explanation.txt"
  fi
  jq --arg method "$method" --arg phase "$phase" --arg status "$status" --arg expected "$expected" \
    --argjson job "$job" --arg url "$AAP_ADDR/execution/jobs/playbook/$job/details" \
    '. + [{method:$method,phase:$phase,job:$job,status:$status,expected:$expected,passed:true,url:$url}]' \
    "$STATE/aap/test-results.json" > "$STATE/aap/test-results.next.json"
  mv "$STATE/aap/test-results.next.json" "$STATE/aap/test-results.json"
  printf '%s / %s: job %s, %s (expected %s)\n' "$method" "$phase" "$job" "$status" "$expected"
}

for method in AppRole OIDC; do
  template=$(jq -er --arg method "$method" '.[$method].job_template' "$STATE/aap/controller-resources.json")
  run_job "$template" "$method" initial successful
done

# Prove that AAP resolves the current KV value at launch; no cached static secret.
openssl rand -hex 32 | tr -d '\n' > "$STATE/aap/demo-password"
vault kv put aap-demo/credentials/test password=@"$STATE/aap/demo-password" purpose=aap-vm-demo >/dev/null
shasum -a 256 "$STATE/aap/demo-password" | awk '{print $1}' > "$STATE/aap/expected-digest"
for method in AppRole OIDC; do
  template=$(jq -er --arg method "$method" '.[$method].job_template' "$STATE/aap/controller-resources.json")
  jq -n --arg method "$method" --rawfile digest "$STATE/aap/expected-digest" \
    '{extra_vars:({aap_auth_method:$method,aap_expected_digest:($digest|rtrimstr("\n"))}|tojson)}' > "$STATE/aap/rotation-payload.json"
  "${AAP_CURL[@]}" -X PATCH --data @"$STATE/aap/rotation-payload.json" "$AAP_API/job_templates/$template/" > "$STATE/aap/template-updated.json"
  run_job "$template" "$method" rotated successful
done

# Copy the authorized template, keeping its OIDC credential but changing its ID.
"${AAP_CURL[@]}" --get --data-urlencode 'name=Vault VM - OIDC rejected template' "$AAP_API/job_templates/" > "$STATE/aap/negative-template-search.json"
NEGATIVE=$(jq -r '.results[0].id // empty' "$STATE/aap/negative-template-search.json")
if [[ -z "$NEGATIVE" ]]; then
  TEMPLATE=$(jq -er .OIDC.job_template "$STATE/aap/controller-resources.json")
  "${AAP_CURL[@]}" -X POST -d '{"name":"Vault VM - OIDC rejected template"}' "$AAP_API/job_templates/$TEMPLATE/copy/" > "$STATE/aap/negative-template.json"
  NEGATIVE=$(jq -er .id "$STATE/aap/negative-template.json")
fi
run_job "$NEGATIVE" OIDC unauthorized_template rejected

jq -n --arg date "$(date -u +%FT%TZ)" --arg aap "$AAP_ADDR" --arg vault "$VAULT_APPLICATION_ADDR" \
  --slurpfile tests "$STATE/aap/test-results.json" \
  '{evaluated_at:$date,aap_url:$aap,vault_url:$vault,tests:$tests[0],passed:($tests[0]|length==5 and all(.passed))}' > "$VM_ROOT/aap/evaluation.json"
jq -e .passed "$VM_ROOT/aap/evaluation.json" >/dev/null


AppRole / initial: job 26, successful (expected successful)


OIDC / initial: job 27, successful (expected successful)


AppRole / rotated: job 28, successful (expected successful)


OIDC / rotated: job 29, successful (expected successful)


OIDC / unauthorized_template: job 30, error (expected rejected)


## Resultados y alcance

`evaluation.json` identifica los jobs y su resultado esperado. El rechazo OIDC es una prueba correcta cuando el job falla por la claim de plantilla. Los fallos de diagnóstico anteriores se conservan en el historial de AAP.

Esta integración cubre consumo de secretos KV v2 mediante AppRole y OIDC de workloads. El acceso web a AAP utiliza el usuario local `admin`. El código no habilita SSO interactivo ni el motor de firma SSH de Vault.

Referencias: [OIDC de AAP](https://docs.redhat.com/en/documentation/red_hat_ansible_automation_platform/2.7/whats_new-oidc_authentication_for_hashicorp_vault), [claims y tipos admitidos por Vault](https://developer.hashicorp.com/vault/api-docs/auth/jwt).

In [9]:
%%bash
set -euo pipefail
source ./aap-env.sh
jq -e '.passed and (.tests | length == 5)' evaluation.json >/dev/null
jq '.tests[] | {method,phase,job,status,expected,passed,url}' evaluation.json
vault status -format=json | jq -e '{initialized,sealed,version} | select(.initialized and (.sealed|not))'


{


  "method": "AppRole",


  "phase": "initial",


  "job": 26,


  "status": "successful",


...

{


  "initialized": true,


  "sealed": false,


  "version": "2.1.1+ent"


}
